<a href="https://colab.research.google.com/github/Blaq19/auspify-ml-internship-task-1/blob/main/Task_1_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

This loads pandas for tables, and scikit-learn for the TF-IDF and similarity maths. Nothing prints.

In [ ]:
df = pd.read_csv("Dataset.csv")
print(df.shape)
df.head()

(8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


You should see (8790, 10) and a table of the first 4 titles.

In [ ]:
df.info()
print(df["type"].value_counts())
print((df == "Not Given").sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB
type
Movie      6126
TV Show    2664
Name: count, dtype: int64
show_id            0
type               0
title              0
director        2588
country          287
date_added         0
release_year       0
rating             0
duration           0
listed_in          0
dtype: int64


This shows that missing values are written as the text “Not Given” (2,588 directors, 287 countries)

In [ ]:
df = df.replace("Not Given", "")
df = df.drop_duplicates(subset=["title", "type", "release_year"]).reset_index(drop=True)
print(df.shape)

(8787, 10)


This blanks out the placeholders and removes 3 duplicate rows. You should see (8787, 10).


In [ ]:
def clean_list(text):
    return " ".join(t.strip().lower().replace(" ", "_") for t in text.split(",") if t.strip())

df["genres_t"]   = df["listed_in"].apply(clean_list)
df["director_t"] = df["director"].apply(clean_list)
df["country_t"]  = df["country"].apply(clean_list)
df["type_t"]     = df["type"].apply(clean_list)
df["rating_t"]   = df["rating"].apply(clean_list)    # new column, original kept

GENRE_WEIGHT = 3     # how many times genres are repeated; try 3 later

df["soup"] = ((df["genres_t"] + " ").str.repeat(GENRE_WEIGHT)
              + df["director_t"] + " "
              + df["country_t"] + " "
              + df["type_t"] + " "
              + df["rating_t"])

df[["title", "rating", "soup"]].head()

,title,rating,soup
0,Dick Johnson Is Dead,PG-13,documentaries documentaries documentaries kirs...
1,Ganglands,TV-MA,crime_tv_shows international_tv_shows tv_actio...
2,Midnight Mass,TV-MA,tv_dramas tv_horror tv_mysteries tv_dramas tv_...
3,Confessions of an Invisible Girl,TV-PG,children_&_family_movies comedies children_&_f...
4,Sankofa,TV-MA,dramas independent_movies international_movies...


Each title now has one line of text describing it.

In [ ]:
tfidf = TfidfVectorizer(token_pattern=r"\S+")
matrix = tfidf.fit_transform(df["soup"])
print(matrix.shape)
print([f for f in tfidf.get_feature_names_out() if f in ("r", "g", "tv-ma", "pg-13")])

(8787, 5130)
['g', 'pg-13', 'r', 'tv-ma']


Each title becomes a row of numbers, like the A, B and C table from my last message. Rare words such as a specific director get more weight than common ones.

In [ ]:
def recommend(title, n=10):
    matches = df.index[df["title"].str.lower() == title.lower()]
    if len(matches) == 0:
        return "Title not found"
    i = matches[0]
    scores = cosine_similarity(matrix[i], matrix).ravel()
    scores[i] = -1                       # don't recommend the title itself
    top = scores.argsort()[::-1][:n]     # indexes of the n highest scores
    result = df.loc[top, ["title", "type", "listed_in", "release_year"]].copy()
    result["similarity"] = scores[top].round(3)
    return result

recommend("Narcos")

,title,type,listed_in,release_year,similarity
8129,Marvel's The Defenders,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",2017,1.000
7941,Marvel's Iron Fist,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",2018,1.000
7667,Marvel's Jessica Jones,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",2019,1.000
7339,Altered Carbon,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",2020,1.000
7216,Queen of the South,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",2018,1.000
7989,Marvel's Luke Cage,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",2018,1.000
8608,Person of Interest,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",2016,0.990
7653,Shooter,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",2018,0.990
7353,Narcos: Mexico,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",2020,0.969
2647,Gotham,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",2019,0.919


this will recommend show that are closely related or similar

In [ ]:
genre_sets = df["listed_in"].apply(lambda s: set(g.strip() for g in s.split(",")))
sample = np.random.RandomState(42).choice(len(df), 300, replace=False)

hits = []
for i in sample:
    scores = cosine_similarity(matrix[i], matrix).ravel()
    scores[i] = -1
    for j in scores.argsort()[::-1][:10]:
        hits.append(len(genre_sets[i] & genre_sets[j]) > 0)
print("Recommendations sharing a genre:", round(np.mean(hits), 3))

rng = np.random.RandomState(0)
rand_hits = [len(genre_sets[i] & genre_sets[j]) > 0
             for i in sample for j in rng.choice(len(df), 10, replace=False)]
print("Random picks sharing a genre:  ", round(np.mean(rand_hits), 3))

Recommendations sharing a genre: 0.998
Random picks sharing a genre:   0.235


The model score is high partly because genres are in the features, so it isn’t a perfect test, but it proves the system works better than random.